# 03 - NIG Heavy-Tail Distribution Fitting

**Project:** ARMA-GARCH Beta Risk Model  
**Author:** Amos Anderson  
**Purpose:** Fit Normal Inverse Gaussian (NIG), Student T, and Gaussian
distributions to the standardised innovations from notebook 02.
Compare tail behaviour across all three models. Generate next-day
return PDFs by combining NIG parameters with GARCH $\sigma $ forecasts.

NIG is the primary model. Student T is retained as a comparison baseline.
Gaussian is included to demonstrate its systematic failure at the tails which is
the core motivation for this entire modelling approach.

In [1]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import kv
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
print("Imports OK")

Imports OK


In [2]:
# Load data

innovations    = load_processed("innovations.parquet")
sigma_forecasts = load_processed("sigma_forecasts.parquet")

# Load full rolling results for actual returns
import os
all_results = {}
TICKERS = list(innovations.columns)

for ticker in TICKERS:
    safe_name = ticker.replace("^", "").replace("=", "_")
    df = load_processed(f"rolling_{safe_name}.parquet")
    all_results[ticker] = df

print(f"\nTickers: {TICKERS}")
print(f"Innovations shape:     {innovations.shape}")
print(f"Sigma forecasts shape: {sigma_forecasts.shape}")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\sigma_forecasts.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_AAPL.parquet  shape=(1000, 3)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_EURUSD_X.parquet  shape=(1000, 3)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Pro

In [20]:
# Implement src/nig.py

from src.nig import (
    NIGParams, nig_pdf, nig_cdf,
    fit_nig_mle, nig_quantile,
    compute_var, compute_cvar,
)
print("src.nig imported OK")

src.nig imported OK


In [22]:
# Reloading the src module

import importlib
import src.nig as _nig
importlib.reload(_nig)
from src.nig import (
    NIGParams, nig_pdf, nig_cdf,
    fit_nig_mle, nig_quantile,
    compute_var, compute_cvar,
)
print("src.nig reloaded OK")

src.nig reloaded OK


In [4]:
# Fit NIG to all assets

from tqdm import tqdm

nig_params = {}

for ticker in TICKERS:
    innov = innovations[ticker].dropna().values
    print(f"Fitting NIG - {ticker} ({len(innov)} innovations)...", end=" ")
    try:
        p = fit_nig_mle(innov)
        nig_params[ticker] = p
        print(f"OK  -->  {p}")
    except RuntimeError as e:
        print(f"FAILED: {e}")

print("\nNIG fitting complete.")

Fitting NIG - AAPL (1000 innovations)... OK  -->  NIGParams(alpha=0.7913, beta=-0.0324, mu=0.0956, delta=0.9602)
Fitting NIG - EURUSD=X (1000 innovations)... OK  -->  NIGParams(alpha=1.2644, beta=0.0209, mu=-0.0316, delta=1.3696)
Fitting NIG - GLD (1000 innovations)... OK  -->  NIGParams(alpha=1.2642, beta=-0.0412, mu=0.1349, delta=1.4349)
Fitting NIG - TIP (1000 innovations)... OK  -->  NIGParams(alpha=1.5184, beta=-0.0407, mu=0.0608, delta=1.4933)
Fitting NIG - ^GSPC (1000 innovations)... OK  -->  NIGParams(alpha=1.4017, beta=-0.3117, mu=0.3698, delta=1.4165)

NIG fitting complete.


In [5]:
# Fit Student T and Gaussian baselines

studentt_params = {}
gaussian_params = {}

for ticker in TICKERS:
    innov = innovations[ticker].dropna().values

    # Student T — fit degrees of freedom, loc, scale
    df_t, loc_t, scale_t = stats.t.fit(innov)
    studentt_params[ticker] = {"df": df_t, "loc": loc_t, "scale": scale_t}

    # Gaussian
    mu_g, sigma_g = innov.mean(), innov.std()
    gaussian_params[ticker] = {"mu": mu_g, "sigma": sigma_g}

    print(f"{ticker:12s}  Student T: df={df_t:.2f}  "
          f"Gaussian: μ={mu_g:.4f} σ={sigma_g:.4f}")

AAPL          Student T: df=3.71  Gaussian: μ=0.0562 σ=1.1101
EURUSD=X      Student T: df=6.05  Gaussian: μ=-0.0090 σ=1.0460
GLD           Student T: df=6.35  Gaussian: μ=0.0881 σ=1.0649
TIP           Student T: df=6.88  Gaussian: μ=0.0208 σ=0.9955
^GSPC         Student T: df=5.78  Gaussian: μ=0.0468 σ=1.0495


----

## Interpreting NIG parameters

| Parameter | Meaning | What to look for |
|-----------|---------|-----------------|
| $\alpha $ (alpha) | Tail heaviness | Smaller = heavier tails. Equities typically 1–5 |
| $\beta $ (beta) | Skewness | Negative = left-skewed (crash risk). Near 0 = symmetric |
| $\mu $ (mu) | Location | Should be near 0 for standardised innovations |
| $\delta $ (delta) | Scale | Should be near 1 for standardised innovations |

A Gaussian is the limiting case as $\alpha \rightarrow \infty$

Student T with $\nu =5$ is a simpler heavy-tail alternative but
cannot be used for dynamic hedging (no finite exponential moments).

In [18]:
# Plot: NIG vs Student T vs Gaussian on innovations

x_grid = np.linspace(-7, 7, 600)
gaussian_ref = stats.norm.pdf(x_grid)
colors_map = dict(zip(TICKERS, px.colors.qualitative.Plotly))

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=TICKERS,
    vertical_spacing=0.12,
    horizontal_spacing=0.06,
)

positions = [(1,1),(1,2),(1,3),(2,1),(2,2)]

for idx, ticker in enumerate(TICKERS):
    row, col = positions[idx]
    innov = innovations[ticker].dropna().values
    p     = nig_params[ticker]
    t_p   = studentt_params[ticker]

    nig_pdf_vals = nig_pdf(x_grid, p)
    t_pdf_vals   = stats.t.pdf(x_grid, df=t_p["df"],
                               loc=t_p["loc"], scale=t_p["scale"])

    # Empirical histogram
    fig.add_trace(go.Histogram(
        x=innov, nbinsx=80, histnorm="probability density",
        name=ticker, marker_color=colors_map[ticker],
        opacity=0.45, showlegend=False,
    ), row=row, col=col)

    # NIG
    fig.add_trace(go.Scatter(
        x=x_grid, y=nig_pdf_vals, mode="lines",
        name="NIG" if idx == 0 else None,
        showlegend=(idx == 0),
        line=dict(color="#e63946", width=2),
    ), row=row, col=col)

    # Student T
    fig.add_trace(go.Scatter(
        x=x_grid, y=t_pdf_vals, mode="lines",
        name="Student T" if idx == 0 else None,
        showlegend=(idx == 0),
        line=dict(color="#f4a261", width=2, dash="dot"),
    ), row=row, col=col)

    # Gaussian
    fig.add_trace(go.Scatter(
        x=x_grid, y=gaussian_ref, mode="lines",
        name="Gaussian" if idx == 0 else None,
        showlegend=(idx == 0),
        line=dict(color="#457b9d", width=1.5, dash="dash"),
    ), row=row, col=col)

    fig.update_xaxes(range=[-7, 7], row=row, col=col)

fig.update_layout(
    title="Innovation Distributions: NIG vs Student T vs Gaussian",
    height=620,
    template="plotly_white",
    legend=dict(orientation="h", x=0.34, y=-0.08),
)
fig.write_html("../outputs/figures/03_distribution_fits.html")
fig.show()
print("Figure saved to outputs/figures/03_distribution_fits.html")

Figure saved to outputs/figures/03_distribution_fits.html


In [12]:
# Tail zoom plot

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Left tail (losses)", "Right tail (gains)"],
    horizontal_spacing=0.08,
)

x_left  = np.linspace(-7, -1.5, 400)
x_right = np.linspace(1.5,  7,  400)

for ticker in TICKERS:
    p   = nig_params[ticker]
    t_p = studentt_params[ticker]

    for x_tail, side in [(x_left, 1), (x_right, 2)]:
        nig_v = nig_pdf(x_tail, p)
        t_v   = stats.t.pdf(x_tail, df=t_p["df"],
                            loc=t_p["loc"], scale=t_p["scale"])
        g_v   = stats.norm.pdf(x_tail)

        fig.add_trace(go.Scatter(
            x=x_tail, y=nig_v, mode="lines",
            name=f"NIG {ticker}",
            line=dict(color=colors_map[ticker], width=1.5),
            showlegend=(side == 1),
        ), row=1, col=side)

        fig.add_trace(go.Scatter(
            x=x_tail, y=g_v, mode="lines",
            name="Gaussian" if (ticker == TICKERS[0] and side == 1) else None,
            showlegend=(ticker == TICKERS[0] and side == 1),
            line=dict(color="black", width=1, dash="dash"),
        ), row=1, col=side)

fig.update_layout(
    title="Tail Behaviour: NIG vs Gaussian (all assets)",
    height=420,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.write_html("../outputs/figures/03_tail_comparison.html")
fig.show()
print("Figure saved to outputs/figures/03_tail_comparison.html")

Figure saved to outputs/figures/03_tail_comparison.html


In [23]:
# QQ Plots: NIG vs Gaussian vs Student T

fig = make_subplots(
    rows=len(TICKERS), cols=3,
    column_titles=["Gaussian QQ", "Student T QQ", "NIG QQ"],
    row_titles=TICKERS,
    vertical_spacing=0.05,
    horizontal_spacing=0.06,
)

quantile_levels = np.linspace(0.01, 0.99, 200)

for i, ticker in enumerate(TICKERS):
    innov  = innovations[ticker].dropna().values
    emp_q  = np.quantile(innov, quantile_levels)
    p      = nig_params[ticker]
    t_p    = studentt_params[ticker]

    # Gaussian theoretical quantiles
    gauss_q = stats.norm.ppf(quantile_levels)

    # Student T theoretical quantiles
    t_q = stats.t.ppf(quantile_levels,
                      df=t_p["df"], loc=t_p["loc"], scale=t_p["scale"])

    # NIG theoretical quantiles (slower — uses bisection CDF)
    nig_q = np.array([nig_quantile(lv, p) for lv in quantile_levels])

    for j, (theory_q, label, color) in enumerate([
        (gauss_q, "Gaussian", "#457b9d"),
        (t_q,     "Student T", "#f4a261"),
        (nig_q,   "NIG",       "#e63946"),
    ]):
        col = j + 1
        fig.add_trace(go.Scatter(
            x=theory_q, y=emp_q, mode="markers",
            marker=dict(color=color, size=3, opacity=0.7),
            name=label, showlegend=(i == 0),
        ), row=i+1, col=col)

        # 45-degree reference line
        lim = max(abs(theory_q.min()), abs(theory_q.max()), 4)
        fig.add_trace(go.Scatter(
            x=[-lim, lim], y=[-lim, lim], mode="lines",
            line=dict(color="black", width=1, dash="dash"),
            showlegend=False,
        ), row=i+1, col=col)

fig.update_layout(
    title="QQ Plots: Gaussian vs Student T vs NIG — All Assets",
    height=300 * len(TICKERS),
    template="plotly_white",
    legend=dict(orientation="h", y=-0.02),
)
fig.write_html("../outputs/figures/03_qq_plots.html")
fig.show()
print("Figure saved to outputs/figures/03_qq_plots.html")

Figure saved to outputs/figures/03_qq_plots.html


In [24]:
# Goodness-of-fit table

from scipy.stats import kstest

rows = []
for ticker in TICKERS:
    innov = innovations[ticker].dropna().values
    p     = nig_params[ticker]
    t_p   = studentt_params[ticker]

    # KS statistic against each distribution
    ks_gauss, _ = kstest(innov, "norm")
    ks_t, _     = kstest(innov, "t",
                         args=(t_p["df"], t_p["loc"], t_p["scale"]))

    # For NIG: use empirical CDF vs NIG CDF
    innov_sorted = np.sort(innov)
    nig_cdf_vals = nig_cdf(innov_sorted, p)
    emp_cdf_vals = np.arange(1, len(innov)+1) / len(innov)
    ks_nig       = float(np.max(np.abs(emp_cdf_vals - nig_cdf_vals)))

    rows.append({
        "Ticker":         ticker,
        "KS - Gaussian":  round(ks_gauss, 4),
        "KS - Student T": round(ks_t, 4),
        "KS - NIG":       round(ks_nig, 4),
        "Best fit":       min(
            [("Gaussian", ks_gauss),
             ("Student T", ks_t),
             ("NIG", ks_nig)],
            key=lambda x: x[1]
        )[0],
    })

gof_df = pd.DataFrame(rows).set_index("Ticker")
print("Goodness of Fit - KS Statistic (lower = better fit)\n")
print(gof_df.to_string())
print("\nNote: KS measures max deviation between empirical and theoretical CDF.")

Goodness of Fit - KS Statistic (lower = better fit)

          KS - Gaussian  KS - Student T  KS - NIG Best fit
Ticker                                                    
AAPL             0.0613          0.0199    0.0155      NIG
EURUSD=X         0.0310          0.0143    0.0119      NIG
GLD              0.0560          0.0142    0.0115      NIG
TIP              0.0436          0.0247    0.0245      NIG
^GSPC            0.0602          0.0181    0.0170      NIG

Note: KS measures max deviation between empirical and theoretical CDF.


In [25]:
# Save NIG Parameters

# Save as DataFrame for use in notebook 04
nig_records = []
for ticker, p in nig_params.items():
    nig_records.append({
        "ticker": ticker,
        "alpha":  p.alpha,
        "beta":   p.beta,
        "mu":     p.mu,
        "delta":  p.delta,
    })

nig_params_df = pd.DataFrame(nig_records).set_index("ticker")
save_processed(nig_params_df, "nig_params.parquet")

print("\nNIG parameters saved:\n")
print(nig_params_df.round(4).to_string())

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\nig_params.parquet

NIG parameters saved:

           alpha    beta      mu   delta
ticker                                  
AAPL      0.7913 -0.0324  0.0956  0.9602
EURUSD=X  1.2644  0.0209 -0.0316  1.3696
GLD       1.2642 -0.0412  0.1349  1.4349
TIP       1.5184 -0.0407  0.0608  1.4933
^GSPC     1.4017 -0.3117  0.3698  1.4165


In [26]:
# Verification

nig_check = load_processed("nig_params.parquet")
assert set(nig_check.index) == set(TICKERS), "Ticker mismatch in saved NIG params"
assert list(nig_check.columns) == ["alpha", "beta", "mu", "delta"], "Column mismatch"
assert (nig_check["alpha"] > 0).all(), "Invalid alpha (must be > 0)"
assert (nig_check["delta"] > 0).all(), "Invalid delta (must be > 0)"
assert (nig_check["alpha"] > nig_check["beta"].abs()).all(), \
    "Constraint violated: alpha must be > |beta|"

print("Verification PASSED — all NIG parameter constraints satisfied")
print(nig_check.round(4).to_string())

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\nig_params.parquet  shape=(5, 4)
Verification PASSED — all NIG parameter constraints satisfied
           alpha    beta      mu   delta
ticker                                  
AAPL      0.7913 -0.0324  0.0956  0.9602
EURUSD=X  1.2644  0.0209 -0.0316  1.3696
GLD       1.2642 -0.0412  0.1349  1.4349
TIP       1.5184 -0.0407  0.0608  1.4933
^GSPC     1.4017 -0.3117  0.3698  1.4165


---

## Handoff to notebook 04

Saved to `data/processed/`:
- `nig_params.parquet` -- fitted NIG parameters ($\alpha, \beta, \mu, \delta $) per asset

**Notebook 04** combines:
- `nig_params.parquet` -- distribution shape per asset
- `sigma_forecasts.parquet` -- GARCH one-step-ahead volatility per day

To produce 1,000 daily VaR and CVaR estimates per asset at both
95% and 99% confidence levels. These are then backtested against
actual returns to count exceedances and compute binomial p-values.